# Follow-up Adequacy
**We examine censoring rates by treatment year to assess whether patients have sufficient follow-up to produce reliable pseudo-observations at 2 years.**

In [1]:
import numpy as np
import pandas as pd

from flatiron_cleaner import DataProcessorRenal

## Import data

In [2]:
dtype_map = pd.read_csv('../outputs/ioio_tki_features_dtypes.csv', index_col = 0).iloc[:, 0].to_dict()
df = pd.read_csv('../outputs/ioio_tki_features.csv', dtype = dtype_map)

In [3]:
df.shape

(4366, 161)

In [4]:
treatment_df = pd.read_csv('../outputs/ioio_tki_index.csv')

In [5]:
treatment_df.shape

(4831, 3)

In [6]:
df = pd.merge(df, treatment_df, on = 'PatientID', how = 'left')

In [7]:
df.shape

(4366, 163)

In [8]:
df['StartDate'] = pd.to_datetime(df['StartDate'])

In [9]:
df['treatment_year'] = df['StartDate'].dt.year

## Start date

In [10]:
df.StartDate.min()

Timestamp('2011-02-04 00:00:00')

## Data lock

In [11]:
# Initialize class 
processor = DataProcessorRenal()

mortality_df = processor.process_mortality(file_path = '../data/Enhanced_Mortality_V2.csv',
                                           index_date_df = df, 
                                           index_date_column = 'StartDate',
                                           visit_path = '../data/Visit.csv', 
                                           telemedicine_path = '../data/Telemedicine.csv', 
                                           biomarkers_path = '../data/Enhanced_MetRCCBiomarkers.csv', 
                                           oral_path = '../data/Enhanced_MetRCC_Orals.csv',
                                           progression_path = '../data/Enhanced_MetRCCProgression.csv',
                                           drop_dates = False)

2026-05-03 20:53:58,350 - INFO - Successfully read Enhanced_Mortality_V2.csv file with shape: (6345, 2) and unique PatientIDs: 6345
2026-05-03 20:53:58,360 - INFO - Successfully merged Enhanced_Mortality_V2.csv df with index_date_df resulting in shape: (4366, 3) and unique PatientIDs: 4366
2026-05-03 20:53:58,642 - INFO - The following columns ['last_visit_date', 'last_biomarker_date', 'last_oral_date', 'last_progression_date'] are used to calculate the last EHR date
2026-05-03 20:53:58,646 - INFO - Successfully processed Enhanced_Mortality_V2.csv file with final shape: (4366, 6) and unique PatientIDs: 4366. There are 0 out of 4366 patients with missing duration values


In [12]:
mortality_df.head(2)

,PatientID,imported_StartDate,DateOfDeath,event,last_ehr_activity,duration
0,F42ACFD609CE6,2015-05-08,NaT,0,2015-12-09,215.0
1,F2A697A9BC9E5,2016-11-02,2019-06-15,1,2019-06-10,955.0


In [13]:
mortality_df['last_date'] = mortality_df[['DateOfDeath', 'last_ehr_activity']].max(axis=1)

In [14]:
data_lock = mortality_df['last_date'].max()

In [15]:
data_lock

Timestamp('2021-10-31 00:00:00')

## Censoring rates by treatment years

In [16]:
df.groupby('treatment_year')['event'].apply(lambda x: (x == 0).mean())

treatment_year
2011    0.174863
2012    0.138686
2013    0.199513
2014    0.227898
2015    0.232990
2016    0.293805
2017    0.377289
2018    0.437247
2019    0.541985
2020    0.637931
2021    0.824074
Name: event, dtype: float64

In [17]:
results = []
for year in sorted(df['treatment_year'].unique()):
    censored = df.query('treatment_year == @year and event == 0', engine='python')
    if len(censored) > 0:
        frac = (censored['duration'] < 1095).mean()
        results.append({'year': year, 'frac_censored_before_2y': frac})

pd.DataFrame(results)

,year,frac_censored_before_2y
0,2011,0.406250
1,2012,0.368421
2,2013,0.414634
3,2014,0.422414
4,2015,0.362832
5,2016,0.397590
6,2017,0.334951
7,2018,0.486111
8,2019,1.000000
9,2020,1.000000


**Primary analysis will be restricted to patients treated in 2020 or earlier.**